In [ ]:
# Resolve repository imports and data paths when opened from notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "sae").is_dir():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Run this notebook from the KronSAE repository.")
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
# We try to check sae that trained on same place 
# We have TopK and KronSAE. We want to check if the features that are correlated in TopK are located in same head in KronSAE.
# for it we can do hard match (1 to 1 feature map) and soft match (for every features we have some candidates).

In [ ]:
# Hypothesis: activation-correlated KronSAE features should fall in the same
# Kron head more often than expected under shuffled head assignments.
# Here KronSAE means mul_fractal_topk / FractalTopKSAE.

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

from sae import KronSAE, SAEConfig, TopKSAE
from safetensors.torch import load_model

sns.set_theme(style="whitegrid")
torch.set_grad_enabled(False)


In [ ]:
MODEL_NAME = "google/gemma-2-2b"
LAYER_NUM = 12
HOOK_POINT = "resid_post"

DATASET_PATH = "HuggingFaceFW/fineweb-edu"
DATASET_NAME = "CC-MAIN-2024-51"
DATASET_SPLIT = "train"

TOPK_CHECKPOINT = "saes/checkpoints_google_gemma-2-2b_blocks.12._d16384_type_topk_l32_t1000000000_h-1_m-1_n-1_suf1000m_exp_2_1b_sweep_no_isoflops_runacfc2077233040c6bfadf0e3bf149773_seed42"
KRON_CHECKPOINT = "saes/checkpoints_google_gemma-2-2b_blocks.12._d16384_type_mul_fractal_topk_l32_t1000000000_h512_m4_n8_suf1000m_exp_2_1b_sweep_no_isoflops_run45c099a2c75b4f8e9dee51316526541a_seed42"
# KRON_CHECKPOINT = "saes/checkpoints_google_gemma-2-2b_blocks.12._d16384_type_mul_fractal_topk_l32_t1000000000_h1024_m1_n16_suf1000m_exp_2_1b_sweep_no_isoflops_run2e54f12255ee47588f12acb6991096d6_seed42"

THRESHOLDS = [0.1]
DEBUG = False

NUM_ITERS = 2 if DEBUG else 500
BATCH_SIZE = 2 if DEBUG else 64
N_CTX = 128 if DEBUG else 1024

NUM_BASELINE_TRIALS = 1000
MAX_HISTOGRAM_SAMPLES = 100_000
RANDOM_SEED = 42
CORR_EPS = 1e-6

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32


In [ ]:
def load_sae(kind, checkpoint, device=DEVICE, force_dtype="float32"):
    classes = {
        "topk": TopKSAE,
        "kron": KronSAE,
        "mulsae": KronSAE,
        "mul_fractal_topk": KronSAE,
    }
    cls = classes[kind.lower()]
    if "PLACEHOLDER" in checkpoint:
        raise ValueError(f"Replace checkpoint placeholder: {checkpoint}")

    checkpoint = Path(checkpoint)
    cfg = SAEConfig.from_pretrained(checkpoint)
    cfg.dtype = force_dtype
    cfg.sae_dtype = force_dtype
    if cls is KronSAE:
        cfg.cartesian_op = "mul"
        missing = [x for x in (cfg.num_heads, cfg.num_mkeys, cfg.num_nkeys) if x is None]
        if missing:
            raise ValueError("KronSAE config must define num_heads, num_mkeys, and num_nkeys")

    sae = cls(cfg)
    load_model(sae, str(checkpoint / "model.safetensors"))
    return {"sae": sae.to(device).eval(), "config": cfg}


model = HookedTransformer.from_pretrained(MODEL_NAME, dtype=MODEL_DTYPE).to(DEVICE).eval()
dataset = load_dataset(DATASET_PATH, name=DATASET_NAME, split=DATASET_SPLIT, streaming=True, trust_remote_code=True)

topk = load_sae("topk", TOPK_CHECKPOINT)
kron = load_sae("kron", KRON_CHECKPOINT)


In [ ]:
def collect_token_batches(model, dataset, num_iters=NUM_ITERS, batch_size=BATCH_SIZE, n_ctx=N_CTX):
    batches, iterator, target = [], iter(dataset), batch_size * n_ctx
    for _ in tqdm(range(num_iters), desc="token batches"):
        chunks, total = [], 0
        while total < target:
            tokens = model.to_tokens(next(iterator)["text"])[0].detach().cpu()
            if tokens.numel():
                chunks.append(tokens)
                total += tokens.numel()
        batches.append(torch.cat(chunks)[:target].reshape(batch_size, n_ctx).long())
    return batches



token_batches = collect_token_batches(model, dataset)
# 
# kron_stats = activation_corr(kron["sae"], model, token_batches)

In [ ]:
token_batches[0].shape

In [ ]:

def activation_corr(sae, model, token_batches, layer_num=LAYER_NUM, hook_point=HOOK_POINT):
    hook_name = utils.get_act_name(hook_point, layer_num)
    state = {"mu": None, "M2": None, "n": 0}

    sae_encode = torch.compile(sae.encode)

    def update_cov(x):
        x = x.float()
        batch_n = x.shape[0]
        old_n = state["n"]
        new_n = old_n + batch_n
        batch_mu = x.mean(0)
        dev = x - batch_mu
        batch_M2 = dev.t().mm(dev)

        if state["mu"] is None:
            state.update(mu=batch_mu, M2=batch_M2, n=batch_n)
            return

        delta = batch_mu - state["mu"]
        state["mu"] += delta * (batch_n / new_n)
        state["M2"] += batch_M2 + torch.outer(delta, delta) * (old_n * batch_n / new_n)
        state["n"] = new_n

    # def hook(resid, _hook):
    #     hidden = sae.encode(resid.reshape(-1, resid.shape[-1]).float().to(DEVICE))
    #     update_cov(hidden.reshape(-1, hidden.shape[-1]))
    #     return resid
    def hook(resid, hook=None):
        hidden = sae_encode(resid.reshape(-1, resid.shape[-1]).float().to(DEVICE))
        update_cov(hidden.reshape(-1, hidden.shape[-1]))
        return resid

    with torch.no_grad():
        for batch in tqdm(token_batches, desc="TopK-Kron cross correlations"):
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                model.run_with_hooks(
                    batch.to(DEVICE),
                    fwd_hooks=[(hook_name, hook)],
                    stop_at_layer=LAYER_NUM + 1,
                    return_type=None,
                )

    cov = state["M2"] / (state["n"] - 1)
    std = torch.sqrt(torch.clamp(torch.diag(cov), min=0.0))
    valid = torch.isfinite(std) & (std > CORR_EPS)
    corr = (cov / (std[:, None] * std[None, :] + CORR_EPS)).clamp(-1, 1)
    corr.fill_diagonal_(0)
    corr[~valid, :] = float("nan")
    corr[:, ~valid] = float("nan")
    return {"corr": corr.cpu(), "std": std.cpu(), "valid": valid.cpu(), "n": state["n"]}
topk_stats = activation_corr(topk["sae"], model, token_batches)

In [ ]:
def topk_kron_cross_corr(topk_sae, kron_sae, model, token_batches):
    hook_name = utils.get_act_name(HOOK_POINT, LAYER_NUM)
    state = {"mx": None, "my": None, "C": None, "M2x": None, "M2y": None, "n": 0}

    topk_sae_encode = torch.compile(topk_sae.encode)
    kron_sae_encode = torch.compile(kron_sae.encode)

    def update(x, y):
        x, y = x.float(), y.float()
        b = x.shape[0]
        old_n = state["n"]
        new_n = old_n + b

        bx, by = x.mean(0), y.mean(0)
        dx, dy = x - bx, y - by

        if state["mx"] is None:
            state["mx"], state["my"] = bx, by
            state["C"] = dx.t().mm(dy)
            state["M2x"] = (dx * dx).sum(0)
            state["M2y"] = (dy * dy).sum(0)
            state["n"] = b
            return

        delta_x = bx - state["mx"]
        delta_y = by - state["my"]
        scale = old_n * b / new_n

        state["mx"] += delta_x * (b / new_n)
        state["my"] += delta_y * (b / new_n)
        state["C"] += dx.t().mm(dy) + torch.outer(delta_x, delta_y) * scale
        state["M2x"] += (dx * dx).sum(0) + delta_x.pow(2) * scale
        state["M2y"] += (dy * dy).sum(0) + delta_y.pow(2) * scale
        state["n"] = new_n

    def hook(resid, hook=None):
        x = resid.reshape(-1, resid.shape[-1]).float().to(DEVICE)
        # topk_h = topk_sae.encode(x).reshape(x.shape[0], -1)
        # kron_h = kron_sae.encode(x).reshape(x.shape[0], -1)
        topk_h = topk_sae_encode(x).reshape(x.shape[0], -1)
        kron_h = kron_sae_encode(x).reshape(x.shape[0], -1)
        update(topk_h, kron_h)
        return resid

    with torch.no_grad():
        for batch in tqdm(token_batches, desc="TopK-Kron cross correlations"):
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                model.run_with_hooks(
                    batch.to(DEVICE),
                    fwd_hooks=[(hook_name, hook)],
                    stop_at_layer=LAYER_NUM + 1,
                    return_type=None,
                )

    std_topk = torch.sqrt(torch.clamp(state["M2x"] / (state["n"] - 1), min=0))
    std_kron = torch.sqrt(torch.clamp(state["M2y"] / (state["n"] - 1), min=0))
    valid_topk = torch.isfinite(std_topk) & (std_topk > CORR_EPS)
    valid_kron = torch.isfinite(std_kron) & (std_kron > CORR_EPS)

    cross_corr = state["C"] / (state["n"] - 1)
    cross_corr = cross_corr / (std_topk[:, None] * std_kron[None, :] + CORR_EPS)
    cross_corr = cross_corr.clamp(-1, 1)
    cross_corr[~valid_topk, :] = float("nan")
    cross_corr[:, ~valid_kron] = float("nan")

    return {
        "corr": cross_corr.cpu(),
        "valid_topk": valid_topk.cpu(),
        "valid_kron": valid_kron.cpu(),
        "n": state["n"],
    }


def kron_head_ids(cfg):
    features_per_head = int(cfg.num_mkeys) * int(cfg.num_nkeys)
    return torch.arange(int(cfg.dict_size)) // features_per_head


def matched_kron_pairs(cross_corr, valid_topk, valid_kron, threshold):
    pairs = []
    active_topk = 0

    for topk_i in torch.where(valid_topk)[0]:
        kron_matches = torch.where(
            torch.isfinite(cross_corr[topk_i])
            & valid_kron
            & (cross_corr[topk_i] > threshold)
        )[0]

        if kron_matches.numel() < 2:
            continue

        active_topk += 1
        a, b = torch.triu_indices(kron_matches.numel(), kron_matches.numel(), offset=1)
        pairs.append(torch.stack([kron_matches[a], kron_matches[b]], dim=1))

    if not pairs:
        return torch.empty((0, 2), dtype=torch.long), active_topk

    # Unique Kron pairs that share at least one TopK feature match.
    return torch.unique(torch.cat(pairs).long(), dim=0), active_topk


def same_head_fraction(pairs, heads):
    if pairs.numel() == 0:
        return np.nan
    return (heads[pairs[:, 0]] == heads[pairs[:, 1]]).float().mean().item()


def shuffled_baseline(pairs, heads, valid_kron, trials=NUM_BASELINE_TRIALS, seed=RANDOM_SEED):
    if pairs.numel() == 0:
        return np.nan, np.nan, np.nan

    heads = torch.as_tensor(heads, dtype=torch.long)
    valid_idx = torch.where(torch.as_tensor(valid_kron, dtype=torch.bool))[0]
    rng = torch.Generator().manual_seed(seed)

    observed = same_head_fraction(pairs, heads)
    values = []

    for _ in range(trials):
        shuffled = heads.clone()
        shuffled[valid_idx] = heads[valid_idx[torch.randperm(valid_idx.numel(), generator=rng)]]
        values.append(same_head_fraction(pairs, shuffled))

    values = np.asarray(values)
    p_value = (np.sum(values >= observed) + 1) / (len(values) + 1)
    return values.mean(), values.std(ddof=1), p_value


def summarize_topk_to_kron(cross_stats, thresholds, kron_heads):
    rows = []

    for threshold in thresholds:
        pairs, active_topk = matched_kron_pairs(
            cross_stats["corr"],
            cross_stats["valid_topk"],
            cross_stats["valid_kron"],
            threshold,
        )

        observed = same_head_fraction(pairs, kron_heads)
        mean, std, p_value = shuffled_baseline(
            pairs,
            kron_heads,
            cross_stats["valid_kron"],
        )

        rows.append({
            "threshold": threshold,
            "model": "TopK -> KronSAE",
            "num_topk_features_with_2plus_kron_matches": active_topk,
            "num_kron_pairs": int(pairs.shape[0]),
            "same_head_fraction": observed,
            "baseline_mean": mean,
            "baseline_std": std,
            "enrichment": observed / mean if mean and not np.isnan(mean) else np.nan,
            "p_value": p_value,
        })

    return pd.DataFrame(rows)


kron_heads = kron_head_ids(kron["config"])
features_per_head = int(kron["config"].num_mkeys) * int(kron["config"].num_nkeys)

assert int(kron_heads.max()) + 1 == int(kron["config"].num_heads)
assert int(kron["config"].dict_size) == int(kron["config"].num_heads) * features_per_head

cross_stats = topk_kron_cross_corr(topk["sae"], kron["sae"], model, token_batches)

# NEW: Do not display the old one-TopK-to-many-Kron analysis here.
# NEW: The intended experiment is in the next cell: correlated TopK pairs -> mapped Kron pairs -> same Kron head.
# old_topk_to_many_kron_df = summarize_topk_to_kron(cross_stats, THRESHOLDS, kron_heads)
# display(old_topk_to_many_kron_df)


In [ ]:
# a = cross_stats['corr']
# torch.where(~a.isnan(),a,0).argmax(dim=1)

In [ ]:
# Simpler experiment:
# 1. Map TopK features to Kron features with a clean 1-to-1 mutual-argmax match.
# 2. For each selected TopK feature, find its most correlated other selected TopK feature.
# 3. Count whether both mapped Kron features are in the same Kron head.
# 4. Baseline: shuffle the TopK->Kron mapping across selected features only.
#    This keeps the TopK nearest-neighbor graph fixed and keeps the same Kron features,
#    but removes the real alignment between TopK correlations and Kron heads.

def kron_head_ids(cfg):
    features_per_head = int(cfg.num_mkeys) * int(cfg.num_nkeys)
    return torch.arange(int(cfg.dict_size)) // features_per_head


def topk_to_kron_one_to_one(
    cross_corr,
    valid_topk,
    valid_kron,
    min_match_corr=0.3,
    min_margin=0.05,
):
    corr = torch.as_tensor(cross_corr).clone()
    # corr = torch.where(corr < 0.01, corr, 0)
    valid_topk = torch.as_tensor(valid_topk, dtype=torch.bool)
    valid_kron = torch.as_tensor(valid_kron, dtype=torch.bool)

    corr[~valid_topk, :] = -float("inf")
    corr[:, ~valid_kron] = -float("inf")
    corr[~torch.isfinite(corr)] = -float("inf")

    top2_corr, top2_kron = corr.topk(2, dim=1)
    best_corr = top2_corr[:, 0]
    second_corr = top2_corr[:, 1]
    best_kron = top2_kron[:, 0]
    margin = best_corr - second_corr

    # Mutual best match gives a simple 1-to-1 map:
    # TopK i chooses Kron j, and Kron j also chooses TopK i.
    best_topk_for_kron = corr.argmax(dim=0)
    topk_ids = torch.arange(corr.shape[0])

    mapped = (
        valid_topk
        & valid_kron[best_kron]
        & torch.isfinite(best_corr)
        & torch.isfinite(second_corr)
        & (best_corr >= min_match_corr)
        & (margin >= min_margin)
        & (best_topk_for_kron[best_kron] == topk_ids)
    )

    assert torch.unique(best_kron[mapped]).numel() == int(mapped.sum())

    return best_kron.long(), best_corr, margin, mapped


def nearest_correlated_topk(topk_corr, selected_topk, min_corr=0.2, chunk_size=512):
    corr = torch.as_tensor(topk_corr)
    selected_topk = torch.as_tensor(selected_topk, dtype=torch.bool)

    src_all, nbr_all, corr_all = [], [], []
    n = corr.shape[0]

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        rows = torch.arange(start, end)

        row_ok = selected_topk[start:end]
        if not row_ok.any():
            continue

        block = corr[start:end].clone()
        block[:, ~selected_topk] = -float("inf")
        block[~row_ok, :] = -float("inf")
        block[torch.arange(end - start), rows] = -float("inf")
        block[~torch.isfinite(block)] = -float("inf")

        best_corr, best_nbr = block.max(dim=1)
        keep = row_ok & torch.isfinite(best_corr) & (best_corr >= min_corr)

        if keep.any():
            src_all.append(rows[keep])
            nbr_all.append(best_nbr[keep].long())
            corr_all.append(best_corr[keep])

    if not src_all:
        empty_long = torch.empty(0, dtype=torch.long)
        empty_float = torch.empty(0, dtype=torch.float32)
        return empty_long, empty_long, empty_float

    return torch.cat(src_all), torch.cat(nbr_all), torch.cat(corr_all)


def same_head_score(src_topk, nbr_topk, best_kron, kron_heads):
    if src_topk.numel() == 0:
        return np.nan, 0

    src_kron = best_kron[src_topk]
    nbr_kron = best_kron[nbr_topk]
    same = kron_heads[src_kron] == kron_heads[nbr_kron]

    s = int(same.sum())
    return float(same.float().mean()), s


def mapping_shuffle_baseline_nn(
    src_topk,
    nbr_topk,
    best_kron,
    kron_heads,
    trials=NUM_BASELINE_TRIALS,
    seed=RANDOM_SEED,
):
    if src_topk.numel() == 0:
        return np.nan, np.nan, np.nan

    # Null hypothesis:
    # The TopK nearest-neighbor graph is real, and the set of matched Kron features is real,
    # but the assignment between selected TopK features and Kron features is random.
    nodes = torch.unique(torch.cat([src_topk, nbr_topk]))
    node_kron = best_kron[nodes]

    node_pos = torch.empty(best_kron.shape[0], dtype=torch.long)
    node_pos[nodes] = torch.arange(nodes.numel())

    src_pos = node_pos[src_topk]
    nbr_pos = node_pos[nbr_topk]

    observed, _ = same_head_score(src_topk, nbr_topk, best_kron, kron_heads)

    rng = torch.Generator().manual_seed(seed)
    values = []

    for _ in range(trials):
        shuffled_node_kron = node_kron[torch.randperm(nodes.numel(), generator=rng)]
        same = kron_heads[shuffled_node_kron[src_pos]] == kron_heads[shuffled_node_kron[nbr_pos]]
        values.append(float(same.float().mean()))

    values = np.asarray(values)
    p_value = (np.sum(values >= observed) + 1) / (len(values) + 1)
    return values.mean(), values.std(ddof=1), p_value


def summarize_nearest_neighbor_same_head(
    topk_stats,
    cross_stats,
    thresholds,
    min_match_corrs,
    kron_heads,
    min_margin=0.05,
):
    rows = []

    for min_match_corr in min_match_corrs:
        best_kron, best_corr, margin, mapped_topk = topk_to_kron_one_to_one(
            cross_stats["corr"],
            cross_stats["valid_topk"],
            cross_stats["valid_kron"],
            min_match_corr=min_match_corr,
            min_margin=min_margin,
        )

        mapped_best_corr = best_corr[mapped_topk]
        mapped_margin = margin[mapped_topk]

        for topk_corr_threshold in thresholds:
            src_topk, nbr_topk, nbr_corr = nearest_correlated_topk(
                topk_stats["corr"],
                topk_stats["valid"] & mapped_topk,
                min_corr=topk_corr_threshold,
            )

            same_head_fraction, s = same_head_score(
                src_topk,
                nbr_topk,
                best_kron,
                kron_heads,
            )

            baseline_mean, baseline_std, p_value = mapping_shuffle_baseline_nn(
                src_topk,
                nbr_topk,
                best_kron,
                kron_heads,
            )

            rows.append({
                "topk_corr_threshold": topk_corr_threshold,
                "min_topk_to_kron_match_corr": min_match_corr,
                "min_match_margin": min_margin,
                "num_1to1_mapped_topk_features": int(mapped_topk.sum()),
                "num_selected_features": int(src_topk.numel()),
                "s_same_head": s,
                "same_head_fraction": same_head_fraction,
                "baseline_mean": baseline_mean,
                "baseline_std": baseline_std,
                "enrichment": same_head_fraction / baseline_mean if baseline_mean else np.nan,
                "p_value": p_value,
                "median_match_corr": float(mapped_best_corr.median()) if mapped_best_corr.numel() else np.nan,
                "median_match_margin": float(mapped_margin.median()) if mapped_margin.numel() else np.nan,
                "median_nearest_topk_corr": float(nbr_corr.median()) if nbr_corr.numel() else np.nan,
            })

    return pd.DataFrame(rows)


kron_heads = kron_head_ids(kron["config"])

features_per_head = int(kron["config"].num_mkeys) * int(kron["config"].num_nkeys)
assert int(kron_heads.max()) + 1 == int(kron["config"].num_heads)
assert int(kron["config"].dict_size) == int(kron["config"].num_heads) * features_per_head

MIN_MATCH_CORRS = [0.1, 0.2, 0.3, 0.4]

results_df = summarize_nearest_neighbor_same_head(
    topk_stats=topk_stats,
    cross_stats=cross_stats,
    thresholds=THRESHOLDS,
    min_match_corrs=MIN_MATCH_CORRS,
    kron_heads=kron_heads,
    min_margin=0.001,
)

display(results_df)


In [ ]:
import numpy as np
import pandas as pd
import torch

# experiment:
# 1. Map TopK features to Kron features with a 1-to-k top k match.
# 2. For each selected TopK feature, find its most correlated other selected TopK features.
# 3. Count whether both mapped Kron features are in the same Kron head.
# 4. Baseline: shuffle the TopK->Kron mapping across selected features only.
#    This keeps the TopK nearest-neighbor graph fixed and keeps the same Kron features,
#    but removes the real alignment between TopK correlations and Kron heads.

def summarize_topr_hits(
    topk_stats,
    cross_stats,
    kron_heads,
    r_values=(1, 2, 3, 4, 5, 10),
    topk_corr_thresholds=(0.1, 0.2, 0.3, 0.4),
    min_match_corr=0.05,
    max_pairs=200_000,
    trials=1000,
    seed=42,
    chunk_size=512,
):
    topk_corr = torch.as_tensor(topk_stats["corr"]).float()
    cross_corr = torch.as_tensor(cross_stats["corr"]).float()
    valid_topk = torch.as_tensor(topk_stats["valid"], dtype=torch.bool) & torch.as_tensor(cross_stats["valid_topk"], dtype=torch.bool)
    valid_kron = torch.as_tensor(cross_stats["valid_kron"], dtype=torch.bool)
    kron_heads = torch.as_tensor(kron_heads, dtype=torch.long)

    n_topk = cross_corr.shape[0]
    rows = []

    def top_r_matches(r):
        top_idx = torch.zeros((n_topk, r), dtype=torch.long)
        top_val = torch.full((n_topk, r), float("nan"))

        for start in range(0, n_topk, chunk_size):
            end = min(start + chunk_size, n_topk)
            block = cross_corr[start:end].clone()
            block[:, ~valid_kron] = -float("inf")
            block[~valid_topk[start:end], :] = -float("inf")
            block[~torch.isfinite(block)] = -float("inf")
            block[block < min_match_corr] = -float("inf")

            vals, idxs = block.topk(r, dim=1)
            ok = torch.isfinite(vals)
            top_idx[start:end] = torch.where(ok, idxs, torch.zeros_like(idxs)).long()
            top_val[start:end] = vals

        top_ok = torch.isfinite(top_val)
        return top_idx, top_ok, top_val, top_ok.any(dim=1)

    def correlated_pairs(selected_topk, threshold):
        srcs, dsts, vals = [], [], []

        for start in range(0, n_topk, chunk_size):
            end = min(start + chunk_size, n_topk)
            rows_i = torch.arange(start, end)

            block = topk_corr[start:end].clone()
            block[:, ~selected_topk] = -float("inf")
            block[~selected_topk[start:end], :] = -float("inf")
            block[~torch.isfinite(block)] = -float("inf")

            # Keep only unique unordered pairs: dst > src.
            block[:, :start] = -float("inf")
            intra = block[:, start:end]
            intra[torch.tril(torch.ones_like(intra, dtype=torch.bool))] = -float("inf")

            keep = block >= threshold
            if keep.any():
                local_src, dst = torch.where(keep)
                srcs.append(rows_i[local_src].long())
                dsts.append(dst.long())
                vals.append(block[local_src, dst].float())

        if not srcs:
            e_long = torch.empty(0, dtype=torch.long)
            e_float = torch.empty(0)
            return e_long, e_long, e_float

        src, dst, val = torch.cat(srcs), torch.cat(dsts), torch.cat(vals)

        if max_pairs is not None and src.numel() > max_pairs:
            keep = torch.argsort(val, descending=True)[:max_pairs]
            src, dst, val = src[keep], dst[keep], val[keep]

        return src, dst, val

    def hit_scores(src, dst, top_idx, top_ok):
        if src.numel() == 0:
            return torch.empty(0)

        src_idx, dst_idx = top_idx[src], top_idx[dst]
        src_ok, dst_ok = top_ok[src], top_ok[dst]

        same_head = kron_heads[src_idx][:, :, None] == kron_heads[dst_idx][:, None, :]
        valid_pair = src_ok[:, :, None] & dst_ok[:, None, :]
        return (same_head & valid_pair).any(dim=(1, 2)).float()

    def shuffle_baseline(src, dst, top_idx, top_ok):
        if src.numel() == 0:
            return np.nan, np.nan, np.nan

        observed = float(hit_scores(src, dst, top_idx, top_ok).mean())

        nodes = torch.unique(torch.cat([src, dst]))
        pos = torch.empty(n_topk, dtype=torch.long)
        pos[nodes] = torch.arange(nodes.numel())

        src_pos, dst_pos = pos[src], pos[dst]
        node_idx, node_ok = top_idx[nodes], top_ok[nodes]

        rng = torch.Generator().manual_seed(seed)
        vals = []

        for _ in range(trials):
            perm = torch.randperm(nodes.numel(), generator=rng)
            vals.append(float(hit_scores(src_pos, dst_pos, node_idx[perm], node_ok[perm]).mean()))

        vals = np.asarray(vals)
        p_value = (np.sum(vals >= observed) + 1) / (len(vals) + 1)
        return vals.mean(), vals.std(ddof=1), p_value

    for r in r_values:
        top_idx, top_ok, top_val, matched = top_r_matches(r)
        selected_topk = valid_topk & matched

        for threshold in topk_corr_thresholds:
            src, dst, pair_corr = correlated_pairs(selected_topk, threshold)
            scores = hit_scores(src, dst, top_idx, top_ok)

            F = int(scores.numel())
            s = int(scores.sum()) if F else 0
            observed = s / F if F else np.nan
            baseline_mean, baseline_std, p_value = shuffle_baseline(src, dst, top_idx, top_ok)

            rows.append({
                "r": r,
                "topk_corr_threshold": threshold,
                "min_topk_to_kron_match_corr": min_match_corr,
                "num_matched_topk_features": int(matched.sum()),
                "num_topk_pairs_F": F,
                "same_head_hits_s": s,
                "top_r_hit_rate_s_over_F": observed,
                "baseline_mean": baseline_mean,
                "baseline_std": baseline_std,
                "enrichment": observed / baseline_mean if baseline_mean and not np.isnan(baseline_mean) else np.nan,
                "p_value": p_value,
                "median_topk_pair_corr": float(pair_corr.median()) if pair_corr.numel() else np.nan,
                "median_best_match_corr": float(torch.nanmedian(top_val[:, 0])),
            })

    return pd.DataFrame(rows)


topr_hit_results_df = summarize_topr_hits(
    topk_stats=topk_stats,
    cross_stats=cross_stats,
    kron_heads=kron_heads,
)

display(topr_hit_results_df)


In [ ]:
def sample_offdiag(corr, valid, max_samples=MAX_HISTOGRAM_SAMPLES, seed=RANDOM_SEED):
    corr = torch.as_tensor(corr)
    ids = torch.where(torch.as_tensor(valid, dtype=torch.bool))[0].cpu().numpy()
    rng = np.random.default_rng(seed)
    values = []
    while len(values) < max_samples:
        i, j = rng.choice(ids, size=2, replace=False)
        value = float(corr[min(i, j), max(i, j)])
        if math.isfinite(value):
            values.append(value)
    return np.asarray(values, dtype=np.float32)


def sample_within_between(corr, groups, valid, max_between=MAX_HISTOGRAM_SAMPLES, seed=RANDOM_SEED):
    corr = torch.as_tensor(corr)
    groups = torch.as_tensor(groups, dtype=torch.long)
    valid = torch.as_tensor(valid, dtype=torch.bool)
    within = []
    for group in torch.unique(groups):
        ids = torch.where((groups == group) & valid)[0]
        if ids.numel() < 2:
            continue
        i, j = torch.triu_indices(ids.numel(), ids.numel(), offset=1)
        values = corr[ids[i], ids[j]]
        within.append(values[torch.isfinite(values)])
    within = torch.cat(within).cpu().numpy() if within else np.array([], dtype=np.float32)

    valid_ids = torch.where(valid)[0].cpu().numpy()
    rng = np.random.default_rng(seed)
    between = []
    while len(between) < max_between:
        i, j = rng.choice(valid_ids, size=2, replace=False)
        if int(groups[i]) == int(groups[j]):
            continue
        value = float(corr[min(i, j), max(i, j)])
        if math.isfinite(value):
            between.append(value)
    return within, np.asarray(between, dtype=np.float32)


def plot_nearest_neighbor_results(results):
    plot_df = results.sort_values('min_topk_to_kron_match_corr')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=180)
    axes[0].errorbar(
        plot_df['min_topk_to_kron_match_corr'],
        plot_df['same_head_fraction'],
        yerr=plot_df['baseline_std'],
        marker='o',
        label='observed',
    )
    axes[0].plot(
        plot_df['min_topk_to_kron_match_corr'],
        plot_df['baseline_mean'],
        marker='o',
        linestyle='--',
        label='shuffled',
    )
    axes[0].set(
        xlabel='minimum TopK-to-Kron match correlation',
        ylabel='same-head fraction',
        title='Nearest-neighbor same-head concentration',
    )
    axes[0].legend()
    axes[1].plot(
        plot_df['min_topk_to_kron_match_corr'],
        plot_df['enrichment'],
        marker='o',
    )
    axes[1].axhline(1, color='black', linestyle='--', linewidth=1)
    axes[1].set(
        xlabel='minimum TopK-to-Kron match correlation',
        ylabel='enrichment',
        title='Enrichment over shuffled mapping',
    )
    fig.tight_layout()
    return fig


def plot_topr_results(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=180)
    for threshold, group in results.groupby('topk_corr_threshold'):
        group = group.sort_values('r')
        label = f'TopK corr. >= {threshold:g}'
        axes[0].plot(group['r'], group['top_r_hit_rate_s_over_F'], marker='o', label=label)
        axes[0].plot(group['r'], group['baseline_mean'], linestyle='--', alpha=0.7)
        axes[1].plot(group['r'], group['enrichment'], marker='o', label=label)
    axes[0].set(xlabel='number of Kron matches r', ylabel='same-head hit rate', title='Top-r same-head hits')
    axes[0].legend()
    axes[1].axhline(1, color='black', linestyle='--', linewidth=1)
    axes[1].set(xlabel='number of Kron matches r', ylabel='enrichment', title='Enrichment over shuffled mapping')
    axes[1].legend()
    fig.tight_layout()
    return fig


def plot_correlation_distributions(topk_stats, kron_stats, kron_heads):
    topk_all = sample_offdiag(topk_stats['corr'], topk_stats['valid'])
    kron_all = sample_offdiag(kron_stats['corr'], kron_stats['valid'])
    kron_within, kron_between = sample_within_between(
        kron_stats['corr'], kron_heads, kron_stats['valid']
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=180)
    bins = np.linspace(
        min(topk_all.min(), kron_all.min()),
        max(topk_all.max(), kron_all.max()),
        50,
    )
    sns.histplot(topk_all, bins=bins, stat='density', element='step', fill=False, label='TopK', ax=axes[0])
    sns.histplot(kron_all, bins=bins, stat='density', element='step', fill=False, label='KronSAE', ax=axes[0])
    axes[0].set(yscale='log', xlabel='Pearson correlation', ylabel='density', title='Overall correlation baseline')
    axes[0].legend()

    bins = np.linspace(
        min(kron_within.min(), kron_between.min()),
        max(kron_within.max(), kron_between.max()),
        50,
    )
    sns.histplot(kron_within, bins=bins, stat='density', element='step', fill=False, label='within head', ax=axes[1])
    sns.histplot(kron_between, bins=bins, stat='density', element='step', fill=False, label='between heads', ax=axes[1])
    axes[1].set(yscale='log', xlabel='Pearson correlation', ylabel='density', title='KronSAE within vs. between heads')
    axes[1].legend()
    fig.tight_layout()
    return fig


In [ ]:
# The new3 notebook computes TopK and cross-model statistics above. Compute the
# KronSAE within-model correlations needed by the attached distribution figure.
kron_stats = activation_corr(kron['sae'], model, token_batches)

nearest_neighbor_figure = plot_nearest_neighbor_results(results_df)
topr_figure = plot_topr_results(topr_hit_results_df)
correlation_distribution_figure = plot_correlation_distributions(
    topk_stats,
    kron_stats,
    kron_heads,
)
plt.show()
